In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")
application = pd.read_csv("../Dataset/application_record.csv")
credit = pd.read_csv("../Dataset/credit_record.csv")


Libraries imported successfully.


In [ ]:
#credit status distribution
credit['STATUS'].value_counts()

STATUS
C    442031
0    383120
X    209230
1     11090
5      1693
2       868
3       320
4       223
Name: count, dtype: int64

In [15]:
#create target variable
credit['TARGET'] = credit['STATUS'].apply(
    lambda x: 0 if x in ['2', '3', '4', '5'] else 1
)

credit[['STATUS', 'TARGET']].head()

,STATUS,TARGET
0,X,1
1,0,1
2,0,1
3,0,1
4,C,1


In [ ]:
#aggregate target by customer id
credit_target = credit.groupby('ID')['TARGET'].min().reset_index()

credit_target.head()

,ID,TARGET
0,5001711,1
1,5001712,1
2,5001713,1
3,5001714,1
4,5001715,1


In [17]:
#merge the datasets
df = application.merge(
    credit_target,
    on='ID',
    how='inner'
)

df.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,TARGET
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,1
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,1
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0,1
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,1
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,1


In [ ]:
#missing values
df.isnull().sum()

ID                         0
CODE_GENDER                0
FLAG_OWN_CAR               0
FLAG_OWN_REALTY            0
CNT_CHILDREN               0
AMT_INCOME_TOTAL           0
NAME_INCOME_TYPE           0
NAME_EDUCATION_TYPE        0
NAME_FAMILY_STATUS         0
NAME_HOUSING_TYPE          0
DAYS_BIRTH                 0
DAYS_EMPLOYED              0
FLAG_MOBIL                 0
FLAG_WORK_PHONE            0
FLAG_PHONE                 0
FLAG_EMAIL                 0
OCCUPATION_TYPE        11323
CNT_FAM_MEMBERS            0
TARGET                     0
dtype: int64

In [22]:
#handling missing values
categorical = df.select_dtypes(include='object').columns

for col in categorical:
    df[col] = df[col].fillna(df[col].mode()[0])
    numerical = df.select_dtypes(include=['int64','float64']).columns

for col in numerical:
    df[col] = df[col].fillna(df[col].median())

In [23]:
#verify missing values
df.isnull().sum()

ID                     0
CODE_GENDER            0
FLAG_OWN_CAR           0
FLAG_OWN_REALTY        0
CNT_CHILDREN           0
AMT_INCOME_TOTAL       0
NAME_INCOME_TYPE       0
NAME_EDUCATION_TYPE    0
NAME_FAMILY_STATUS     0
NAME_HOUSING_TYPE      0
DAYS_BIRTH             0
DAYS_EMPLOYED          0
FLAG_MOBIL             0
FLAG_WORK_PHONE        0
FLAG_PHONE             0
FLAG_EMAIL             0
OCCUPATION_TYPE        0
CNT_FAM_MEMBERS        0
TARGET                 0
dtype: int64

In [24]:
#remove duplicate records
print("Duplicates Before :", df.duplicated().sum())

df.drop_duplicates(inplace=True)

print("Duplicates After  :", df.duplicated().sum())

Duplicates Before : 0
Duplicates After  : 0


In [25]:
#drop unnecessary columns
df.drop(columns=['ID'], inplace=True)

df.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,TARGET
0,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,Laborers,2.0,1
1,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,Laborers,2.0,1
2,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0,1
3,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,1
4,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,1


In [26]:
#encode categorical features
encoder = LabelEncoder()

categorical_columns = df.select_dtypes(include='object').columns

for column in categorical_columns:
    df[column] = encoder.fit_transform(df[column])

print("Encoding Completed")

Encoding Completed


In [27]:
X = df.drop('TARGET', axis=1)

y = df['TARGET']

In [28]:
#feature scaling
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [29]:
#final processed data
processed_df = pd.DataFrame(
    X_scaled,
    columns=X.columns
)

processed_df['TARGET'] = y.values

processed_df.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,TARGET
0,1.425224,1.278126,0.698338,-0.579661,2.365845,0.923758,-1.563576,-1.433026,2.855131,0.945169,-0.463532,0.0,1.853127,-0.646578,-0.313952,0.056606,-0.217680,1
1,1.425224,1.278126,0.698338,-0.579661,2.365845,0.923758,-1.563576,-1.433026,2.855131,0.945169,-0.463532,0.0,1.853127,-0.646578,-0.313952,0.056606,-0.217680,1
2,1.425224,1.278126,0.698338,-0.579661,-0.728827,0.923758,0.673104,-0.385405,-0.297250,-1.309091,-0.438774,0.0,-0.539628,-0.646578,-0.313952,2.274732,-0.217680,1
3,-0.701644,-0.782396,0.698338,-0.579661,0.818509,-1.383035,0.673104,1.709838,-0.297250,-0.746300,-0.452700,0.0,-0.539628,1.546603,3.185203,1.720201,-1.314564,1
4,-0.701644,-0.782396,0.698338,-0.579661,0.818509,-1.383035,0.673104,1.709838,-0.297250,-0.746300,-0.452700,0.0,-0.539628,1.546603,3.185203,1.720201,-1.314564,1


In [30]:
#save processed data
processed_df.to_csv(
    "../Dataset/processed_data.csv",
    index=False
)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!


In [31]:
saved = pd.read_csv("../Dataset/processed_data.csv")

saved.head()

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,TARGET
0,1.425224,1.278126,0.698338,-0.579661,2.365845,0.923758,-1.563576,-1.433026,2.855131,0.945169,-0.463532,0.0,1.853127,-0.646578,-0.313952,0.056606,-0.217680,1
1,1.425224,1.278126,0.698338,-0.579661,2.365845,0.923758,-1.563576,-1.433026,2.855131,0.945169,-0.463532,0.0,1.853127,-0.646578,-0.313952,0.056606,-0.217680,1
2,1.425224,1.278126,0.698338,-0.579661,-0.728827,0.923758,0.673104,-0.385405,-0.297250,-1.309091,-0.438774,0.0,-0.539628,-0.646578,-0.313952,2.274732,-0.217680,1
3,-0.701644,-0.782396,0.698338,-0.579661,0.818509,-1.383035,0.673104,1.709838,-0.297250,-0.746300,-0.452700,0.0,-0.539628,1.546603,3.185203,1.720201,-1.314564,1
4,-0.701644,-0.782396,0.698338,-0.579661,0.818509,-1.383035,0.673104,1.709838,-0.297250,-0.746300,-0.452700,0.0,-0.539628,1.546603,3.185203,1.720201,-1.314564,1
